In [1]:
!python -V

Python 3.12.1


In [2]:
import pandas as pd

In [3]:
import pickle

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error

In [6]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/workspaces/zoomcamp_MLops_launch/mlruns/1', creation_time=1788126385169, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788126385169, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [7]:
def read_data(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [8]:
df_train =  read_data('02-Experiments/data/green_tripdata_2021-01.parquet')
df_val = read_data('02-Experiments/data/green_tripdata_2021-02.parquet')

In [9]:
len(df_train), len(df_val)

(73908, 61921)

In [10]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [2]:
import numpy as np
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

X_val.indices = X_val.indices.astype(np.int32)
X_val.indptr = X_val.indptr.astype(np.int32)

NameError: name 'DictVectorizer' is not defined

In [12]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [13]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

mean_squared_error(y_val, y_pred)

60.19766169579813

In [14]:
import os
import pickle

os.makedirs('models', exist_ok=True)



with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [15]:
with mlflow.start_run():

    mlflow.set_tag("developer", "cristian")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [16]:
import xgboost as xgb

In [17]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [7]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

NameError: name 'xgb' is not defined

In [19]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [20]:
search_space = {
    # Narrow depth to a specific effective range (e.g., 4 to 6 instead of 3 to 8)
    'max_depth': scope.int(hp.quniform('max_depth', 4, 6, 1)),
    
    'learning_rate': 0.01,
    
 
    
    # Tighten min_child_weight range
    'min_child_weight': hp.loguniform('min_child_weight', 1, 2),
    
    'objective': 'reg:squarederror',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials()
)

  0%|          | 0/10 [00:00<?, ?trial/s, best loss=?]

[0]	validation-rmse:12.13379                          
[1]	validation-rmse:12.05548                          
[2]	validation-rmse:11.97816                          
[3]	validation-rmse:11.90187                          
[4]	validation-rmse:11.82656                          
[5]	validation-rmse:11.75223                          
[6]	validation-rmse:11.67883                          
[7]	validation-rmse:11.60640                          
[8]	validation-rmse:11.53493                          
[9]	validation-rmse:11.46439                          
[10]	validation-rmse:11.39477                         
[11]	validation-rmse:11.32611                         
[12]	validation-rmse:11.25830                         
[13]	validation-rmse:11.19145                         
[14]	validation-rmse:11.12546                         
[15]	validation-rmse:11.06038                         
[16]	validation-rmse:10.99609                         
[17]	validation-rmse:10.93266                         
[18]	valid

In [22]:
mlflow.xgboost.autolog(disable=True)

In [27]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.3675291183024861,
        'max_depth': 27,
        'min_child_weight': 0.8819752466721981,
        'objective': 'reg:linear',
        'reg_alpha': 0.014857162853299723,
        'reg_lambda': 0.19325102202589114,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=100,
        evals=[(valid, 'validation')],
        early_stopping_rounds=25
    )

    y_pred = booster.predict(valid)
    rmse = mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/usr/local/python/3.12.1/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [09:34:58] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:9.42624
[1]	validation-rmse:7.99267
[2]	validation-rmse:7.28394
[3]	validation-rmse:6.95173
[4]	validation-rmse:6.78616
[5]	validation-rmse:6.69817
[6]	validation-rmse:6.64787
[7]	validation-rmse:6.62020
[8]	validation-rmse:6.60655
[9]	validation-rmse:6.59244
[10]	validation-rmse:6.58539
[11]	validation-rmse:6.57942
[12]	validation-rmse:6.57594
[13]	validation-rmse:6.57153
[14]	validation-rmse:6.56667
[15]	validation-rmse:6.56365
[16]	validation-rmse:6.56160
[17]	validation-rmse:6.55786
[18]	validation-rmse:6.55229
[19]	validation-rmse:6.55118
[20]	validation-rmse:6.54594
[21]	validation-rmse:6.54264
[22]	validation-rmse:6.54073
[23]	validation-rmse:6.53908
[24]	validation-rmse:6.53753
[25]	validation-rmse:6.53600
[26]	validation-rmse:6.53503
[27]	validation-rmse:6.53338
[28]	validation-rmse:6.53134
[29]	validation-rmse:6.53098
[30]	validation-rmse:6.52953
[31]	validation-rmse:6.52743
[32]	validation-rmse:6.52611
[33]	validation-rmse:6.52491
[34]	validation-rmse:6.5

2026/09/01 09:35:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


TypeError: sparse array length is ambiguous; use getnnz() or shape[0]